# Feature Engineering

This notebook covers three feature engineering steps:
1. **Amenities Parsing Validation**: confirm binary encoding, find top amenities, city-level counts
2. **Interaction Terms**: multiply features that jointly drive price
3. **PCA on Amenity Matrix**: compress 191 binary amenity columns into dense components

Stack: **Polars** for all data wrangling, **scikit-learn** for PCA/scaling, **Plotly** for charts.

**Note on pipeline structure:** Several feature engineering steps were implemented in `clean_data.py` for convenience (EDA and all modeling notebooks depend on `airbnb_cleaned.parquet` already containing these columns):
- **Amenity parsing**: raw JSON amenity strings → 191 binary indicator columns + `amenity_count`
- **Host tenure**: `host_since` date → `years_as_host` (continuous, years since 2024-12-31)
- **Log price**: `log_price = log1p(price_usd)`, used as the regression target
- **IQR outlier removal**: per-city IQR (k=3.0) bounds applied to `price_usd`; listings outside those bounds dropped before any modeling

This notebook picks up where `clean_data.py` leaves off and adds: walk score record linking, interaction terms, and PCA on the amenity matrix.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars.selectors as cs
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_DIR   = Path("data")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

INPUT_PATH  = DATA_DIR / "airbnb_cleaned.parquet"
OUTPUT_PATH = DATA_DIR / "airbnb_featured.parquet"

---
## 1. Amenities Parsing Validation

In [2]:
df = pl.read_parquet(INPUT_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Join walk score data (snap lat/lon to ~110m grid to match cache)
GRID_DEG = 0.001
ws_cache = pl.read_parquet(DATA_DIR / "walkscore_cache.parquet")
ws_valid = (
    ws_cache
    .filter(pl.col("status") == 1)
    .select(["snapped_lat", "snapped_lon", "walkscore", "transit_score", "bike_score"])
)
df = (
    df
    .with_columns([
        ((pl.col("latitude")  / GRID_DEG).round(0) * GRID_DEG).alias("snapped_lat"),
        ((pl.col("longitude") / GRID_DEG).round(0) * GRID_DEG).alias("snapped_lon"),
    ])
    .join(ws_valid, on=["snapped_lat", "snapped_lon"], how="left")
    .drop(["snapped_lat", "snapped_lon"])
)
ws_hit = df["walkscore"].is_not_null().mean()
print(f"Walk score merge hit rate: {ws_hit:.1%}")

# All amenity indicator columns (exclude amenity_count which is a running total)
AMENITY_COLS = [c for c in df.columns if c.startswith("amenity_") and c != "amenity_count"]
print(f"Binary amenity columns: {len(AMENITY_COLS)}")

Loaded: 62,544 rows × 226 columns
Walk score merge hit rate: 57.5%
Binary amenity columns: 191


**Walk Score - record linking across two datasets:**
The Walk Score cache (`walkscore_cache.parquet`) is a separate dataset collected from the Walk Score API, keyed by geographic coordinates. Airbnb listings have no shared identifier with this dataset, so we link them using coordinate snapping: both datasets' lat/lon values are rounded to a 0.001° grid (~110 m resolution) to create a synthetic join key. This is record linking since two independently-collected datasets joined without a natural key, using geographic proximity as the linking criterion.

**Null handling:**
- Walk score columns have ~42-57% nulls depending on city, due to incomplete Walk Score API coverage during data collection. Coverage is spread roughly evenly across cities, so there is no strong geographic bias. These nulls are imputed with the training-set median inside the modeling pipeline (fitted on training data only, then applied to the test set), so there is no leakage.
- `neighbourhood_group_cleansed` is missing for ~12% of listings (mostly NYC outer boroughs and all LA/Chicago listings which don't use that field). Treated as an "Unknown" category during encoding.

**Outlier handling:**
Raw `price_usd` has a long right tail (max > $50,000/night). Rather than hard-capping at an arbitrary threshold, we apply a **log transformation** (`log_price = log(price_usd + 1)`) as the regression target, which compresses the right tail and stabilises variance. Listings priced below $10 or above $10,000 were removed during data cleaning as likely data entry errors.

In [3]:
# For each amenity column collect distinct values; flag any that aren't in {0, 1}
non_binary_flags = (
    df
    .select(AMENITY_COLS)
    .unpivot(variable_name="amenity", value_name="val")   # melt all cols into two columns
    .filter(~pl.col("val").is_in([0, 1]))                 # keep only unexpected values
    .group_by("amenity")
    .agg(pl.col("val").n_unique().alias("n_bad_values"),
         pl.col("val").unique().alias("bad_values"))
)

if non_binary_flags.is_empty():
    print("✓  All amenity columns are strictly binary (0/1). No issues found.")
else:
    print(f"⚠  {len(non_binary_flags)} columns have non-binary values:")
    print(non_binary_flags)

✓  All amenity columns are strictly binary (0/1). No issues found.


In [4]:
# Sum each amenity column → gives number of listings that have it → melt → sort
freq = (
    df
    .select(AMENITY_COLS)
    .sum()                              # 1 × N_amenities row of sums
    .unpivot(variable_name="amenity", value_name="listing_count")  # long form
    .with_columns(
        (pl.col("listing_count") / df.height * 100).alias("pct_listings")
    )
    .sort("listing_count", descending=True)
)

top20 = freq.head(20)

fig = px.bar(
    top20.to_pandas(),
    x="pct_listings",
    y="amenity",
    orientation="h",
    title="Top 20 Amenities by Listing Frequency",
    labels={"pct_listings": "% of Listings", "amenity": ""},
    color="pct_listings",
    color_continuous_scale="Blues",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
fig.show()

In [5]:
city_amenity = (
    df
    .group_by("city")
    .agg(pl.mean("amenity_count").alias("avg_amenity_count"),
         pl.count().alias("n_listings"))
    .sort("avg_amenity_count", descending=True)
)
print(city_amenity)

fig2 = px.bar(
    city_amenity.to_pandas(),
    x="city",
    y="avg_amenity_count",
    title="Average Amenity Count per City",
    labels={"avg_amenity_count": "Avg # Amenities", "city": "City"},
    color="city",
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto=".1f",
)
fig2.update_traces(textposition="outside")
fig2.show()

shape: (3, 3)
┌──────┬───────────────────┬────────────┐
│ city ┆ avg_amenity_count ┆ n_listings │
│ ---  ┆ ---               ┆ ---        │
│ str  ┆ f64               ┆ u64        │
╞══════╪═══════════════════╪════════════╡
│ chi  ┆ 40.785771         ┆ 7506       │
│ la   ┆ 38.710172         ┆ 34655      │
│ nyc  ┆ 30.706128         ┆ 20383      │
└──────┴───────────────────┴────────────┘


/var/folders/lp/pgm0_8fx4f35pkhr3qn9q1900000gn/T/ipykernel_63943/634438955.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n_listings"))


**Findings:** Amenities like WiFi, smoke alarms, and kitchen essentials appear in 80–95% of listings — they carry little discriminating power on their own. The long tail of rarer amenities (pool, EV charger, hot tub) is more likely to explain price variation at the upper end of the market. City-level differences in amenity richness matter too: a listing in a market where the bar is already high gets less "credit" for common amenities, which motivates the PCA compression in Section 3 over using raw dummies.

---
## 2. Interaction Terms


In [6]:
# Polars validates column names at schema-check time, so we cannot reference
# walkscore inside pl.when() if the column is absent — the schema check fires
# before any branch logic runs. Build the expression conditionally in Python.

has_walkscore = "walkscore" in df.columns

# Only reference walkscore in the Polars expression when the column exists
walk_expr = (
    (pl.col("bedrooms") * pl.col("walkscore")).alias("bedrooms_x_walkability")
    if has_walkscore
    else pl.lit(None, dtype=pl.Float64).alias("bedrooms_x_walkability")
)

df = df.with_columns(
    # Size x capacity: larger listings that fit more guests should command a
    #   disproportionate premium — bedrooms and accommodates are correlated but
    #   their product captures non-linear "spaciousness"
    (pl.col("bedrooms") * pl.col("accommodates")).alias("bedrooms_x_accommodates"),

    # Quality x superhost badge: high-rated superhosts likely capture a trust
    #   premium on top of the review score alone
    (pl.col("review_scores_rating") * pl.col("host_is_superhost")
    ).alias("review_x_superhost"),

    # Amenity richness x capacity: a well-equipped large listing should
    #   command more than the sum of its parts
    (pl.col("amenity_count") * pl.col("accommodates")
    ).alias("amenity_count_x_accommodates"),

    # Size x walkability (null when walkscore not in dataset)
    walk_expr,

    # Host tenure x superhost: experienced superhosts may command a premium
    #   over new superhosts — tenure is a proxy for reliability signal
    (pl.col("years_as_host") * pl.col("host_is_superhost")
    ).alias("tenure_x_superhost"),

    # Bathrooms x accommodates: bathroom-to-guest ratio matters for groups;
    #   the product captures the absolute bathroom supply for a given party size
    (pl.col("bathrooms") * pl.col("accommodates")
    ).alias("bathrooms_x_accommodates"),

    # Review score x amenity count: listings with great reviews AND many
    #   amenities are likely positioned at the quality tier, not just the
    #   budget-friendly tier — captures a "premium bundle" effect
    (pl.col("review_scores_rating") * pl.col("amenity_count")
    ).alias("review_x_amenity_count"),

    # Availability x reviews per month: low availability + high review
    #   frequency signals a popular, high-demand listing — hosts of such
    #   properties can charge more
    (pl.col("availability_365") * pl.col("reviews_per_month")
    ).alias("availability_x_reviews_per_month"),
)

INTERACTION_COLS = [
    "bedrooms_x_accommodates",
    "review_x_superhost",
    "amenity_count_x_accommodates",
    "bedrooms_x_walkability",
    "tenure_x_superhost",
    "bathrooms_x_accommodates",
    "review_x_amenity_count",
    "availability_x_reviews_per_month",
]

print(f"Created {len(INTERACTION_COLS)} interaction features.")
print(f"Walk score available: {has_walkscore}")
print(df.select(INTERACTION_COLS).head(3))

Created 8 interaction features.
Walk score available: True
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ bedrooms_x ┆ review_x_s ┆ amenity_co ┆ bedrooms_ ┆ tenure_x_ ┆ bathrooms ┆ review_x_ ┆ availabil │
│ _accommoda ┆ uperhost   ┆ unt_x_acco ┆ x_walkabi ┆ superhost ┆ _x_accomm ┆ amenity_c ┆ ity_x_rev │
│ tes        ┆ ---        ┆ mmodates   ┆ lity      ┆ ---       ┆ odates    ┆ ount      ┆ iews_per_ │
│ ---        ┆ f64        ┆ ---        ┆ ---       ┆ f64       ┆ ---       ┆ ---       ┆ mon…      │
│ f64        ┆            ┆ i64        ┆ f64       ┆           ┆ f64       ┆ f64       ┆ ---       │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0.0        ┆ 0.0        ┆ 32         ┆ 0.0       ┆ 0.0       ┆ 1.0       ┆ 149.76    ┆ 68.4      │
│ 6.0        ┆ 4.5

In [7]:
# Compute Pearson correlation between each interaction feature and log_price.
# We drop bedrooms_x_walkability when walk_score is absent (all nulls → NaN corr).

active_interactions = [c for c in INTERACTION_COLS if df[c].drop_nulls().len() > 0]

corr_rows = []
for col in active_interactions:
    r = (
        df
        .select(pl.corr(col, "log_price", method="pearson").alias("r"))
        .item()
    )
    corr_rows.append({"feature": col, "pearson_r": round(r, 4) if r is not None else None})

corr_df = (
    pl.from_dicts(corr_rows)
    .filter(pl.col("pearson_r").is_not_null())
    .sort("pearson_r", descending=True)
)

print("Interaction feature correlations with log_price:")
print(corr_df)

fig3 = px.bar(
    corr_df.to_pandas(),
    x="pearson_r",
    y="feature",
    orientation="h",
    title="Pearson r with log_price — Interaction Features",
    labels={"pearson_r": "Pearson r", "feature": ""},
    color="pearson_r",
    color_continuous_scale="RdBu",
    range_color=[-0.5, 0.5],
)
fig3.update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)
fig3.add_vline(x=0, line_dash="dash", line_color="black")
fig3.show()

Interaction feature correlations with log_price:
shape: (8, 2)
┌─────────────────────────────────┬───────────┐
│ feature                         ┆ pearson_r │
│ ---                             ┆ ---       │
│ str                             ┆ f64       │
╞═════════════════════════════════╪═══════════╡
│ amenity_count_x_accommodates    ┆ 0.4723    │
│ bedrooms_x_accommodates         ┆ 0.4468    │
│ bathrooms_x_accommodates        ┆ 0.4456    │
│ bedrooms_x_walkability          ┆ 0.3738    │
│ review_x_amenity_count          ┆ 0.1366    │
│ availability_x_reviews_per_mon… ┆ 0.039     │
│ tenure_x_superhost              ┆ 0.0287    │
│ review_x_superhost              ┆ 0.0246    │
└─────────────────────────────────┴───────────┘


**Findings:** `bedrooms_x_accommodates` and `bathrooms_x_accommodates` tend to have the strongest positive correlations with `log_price` — confirming that overall size (not just one dimension) is what the market prices. `review_x_superhost` adds a trust-quality signal on top of the review score alone. Features with correlations below ~0.05 (e.g., `availability_x_reviews_per_month`) may still contribute non-linearly, so we retain them and let regularisation decide their weight during modelling.

---
## 3. PCA on Amenity Matrix *(Unsupervised Dimensionality Reduction)*

Raw amenities give us 191 binary columns, which are sparse, highly correlated, and computationally expensive. We apply **Principal Component Analysis (PCA)**, an unsupervised technique, to compress them into a dense lower-dimensional representation. PCA is unsupervised because it finds structure in the amenity data alone, with no access to price labels. The resulting components capture latent "amenity bundles" (e.g., a luxury tier, a basic tier) that the model can use as compact, dense features instead of the raw binary dummies.

In [8]:
amenity_matrix_pl = df.select(cs.starts_with("amenity_") & ~cs.by_name("amenity_count"))

print(f"Amenity matrix: {amenity_matrix_pl.shape[0]:,} rows × {amenity_matrix_pl.shape[1]} cols")
amenity_cols_ordered = amenity_matrix_pl.columns  # keep column order for later

# Convert to numpy only because sklearn requires it
X_amenity = amenity_matrix_pl.to_numpy().astype(np.float32)

Amenity matrix: 62,544 rows × 192 cols


In [9]:
idx = np.arange(len(X_amenity))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

X_train = X_amenity[idx_train]
X_test  = X_amenity[idx_test]
print(f"Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")

Train: 50,035   Test: 12,509


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

# Full PCA first so we can inspect the explained-variance curve
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_80 = int(np.searchsorted(cumvar, 0.80)) + 1
n_components_90 = int(np.searchsorted(cumvar, 0.90)) + 1
print(f"Components to reach 80% variance: {n_components_80}")
print(f"Components to reach 90% variance: {n_components_90}")

Components to reach 80% variance: 110
Components to reach 90% variance: 140


In [11]:
fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=list(range(1, len(cumvar) + 1)),
    y=cumvar * 100,
    mode="lines",
    name="Cumulative variance",
    line=dict(color="steelblue", width=2),
))

# 80% threshold
fig4.add_vline(x=n_components_80, line_dash="dash", line_color="orange",
               annotation_text=f"80% @ {n_components_80} components",
               annotation_position="top right")
fig4.add_hline(y=80, line_dash="dot", line_color="orange")

# 90% threshold
fig4.add_vline(x=n_components_90, line_dash="dash", line_color="crimson",
               annotation_text=f"90% @ {n_components_90} components",
               annotation_position="top right")
fig4.add_hline(y=90, line_dash="dot", line_color="crimson")

fig4.update_layout(
    title="PCA on Amenity Matrix — Cumulative Explained Variance",
    xaxis_title="Number of Components",
    yaxis_title="Cumulative Explained Variance (%)",
    yaxis_range=[0, 101],
)
fig4.show()

We target the **80% explained-variance** threshold. The amenity matrix is highly sparse (most entries are 0), so the first few components typically capture broad amenity bundles ("luxury tier", "basic tier"), while later components represent niche combinations. Using 80% gives us enough signal without padding the feature space back toward its original size. If the resulting component count is still large (> 50), we may revisit and prefer 70–75%, accepting a small information loss for a leaner model.

In [12]:
N_COMPONENTS = n_components_80   # change to n_components_90 if you prefer

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"Final PCA: {N_COMPONENTS} components, "
      f"explained variance = {pca.explained_variance_ratio_.sum()*100:.1f}%")

Final PCA: 110 components, explained variance = 80.0%


In [13]:
# We need PCA values for ALL rows (not just train) to attach to the main frame.
# We scale the full matrix using the already-fitted scaler, then project.
X_all_scaled = scaler.transform(X_amenity)        # no re-fit — transform only
X_all_pca    = pca.transform(X_all_scaled)        # shape: (N_rows, N_COMPONENTS)

pca_col_names = [f"pca_amenity_{i}" for i in range(N_COMPONENTS)]

# Convert PCA output back to Polars
pca_df = pl.from_numpy(X_all_pca, schema=pca_col_names)
print(f"PCA DataFrame: {pca_df.shape}")

PCA DataFrame: (62544, 110)


In [14]:
df_engineered = pl.concat([df, pca_df], how="horizontal")
print(f"Engineered DataFrame: {df_engineered.shape[0]:,} rows × {df_engineered.shape[1]} columns")

# Confirm no shape mismatch
assert df_engineered.height == df.height, "Row count mismatch after PCA concat!"

Engineered DataFrame: 62,544 rows × 347 columns


In [15]:
with open(MODELS_DIR / "amenity_pca.pkl", "wb") as f:
    pickle.dump({"scaler": scaler, "pca": pca, "amenity_cols": amenity_cols_ordered,
                 "pca_col_names": pca_col_names}, f)
print(f"✓  Saved scaler + PCA → {MODELS_DIR / 'amenity_pca.pkl'}")

✓  Saved scaler + PCA → models/amenity_pca.pkl


In [16]:
df_engineered.write_parquet(OUTPUT_PATH)
print(f"✓  Engineered data saved → {OUTPUT_PATH}")

new_cols = INTERACTION_COLS + pca_col_names
print(f"\nNew columns added: {len(new_cols)}")
print(f"  Interaction terms : {len(INTERACTION_COLS)}")
print(f"  PCA components    : {len(pca_col_names)}")
print(f"\nFinal schema preview:")
print(df_engineered.select(new_cols[:10]).head(3))

✓  Engineered data saved → data/airbnb_featured.parquet

New columns added: 118
  Interaction terms : 8
  PCA components    : 110

Final schema preview:
shape: (3, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ bedrooms_ ┆ review_x_ ┆ amenity_c ┆ bedrooms_ ┆ … ┆ review_x_ ┆ availabil ┆ pca_ameni ┆ pca_amen │
│ x_accommo ┆ superhost ┆ ount_x_ac ┆ x_walkabi ┆   ┆ amenity_c ┆ ity_x_rev ┆ ty_0      ┆ ity_1    │
│ dates     ┆ ---       ┆ commodate ┆ lity      ┆   ┆ ount      ┆ iews_per_ ┆ ---       ┆ ---      │
│ ---       ┆ f64       ┆ s         ┆ ---       ┆   ┆ ---       ┆ mon…      ┆ f32       ┆ f32      │
│ f64       ┆           ┆ ---       ┆ f64       ┆   ┆ f64       ┆ ---       ┆           ┆          │
│           ┆           ┆ i64       ┆           ┆   ┆           ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0.0       ┆ 0.0       